# Grokking modular addition

Partial replication of [Progress Measures for Grokking via Mechanistic Interpretability](https://arxiv.org/abs/2301.05217) (Nanda et al., 2023).

Goal: train a tiny 1-layer transformer to compute `(a + b) mod 113`, watch it memorise for thousands of steps and then suddenly generalise (the 'grokking' phenomenon), then peek inside the trained weights and find that it learnt a Fourier-based algorithm.

This notebook is the runnable companion to the walkthrough above.

Expected runtime on Colab T4:for the headline experiment.

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

torch.manual_seed(0)
np.random.seed(0)

## 2. Hyperparameters

All knobs in one place so you can experiment. These values are essentially Neel Nanda's defaults from his grokking demo - they're known to reliably produce grokking.

In [ ]:
# task
p = 113                # prime modulus; vocab will be size p+1 (numbers + '=' token)

# model
d_model = 128          # width of the residual stream
n_heads = 4            # number of attention heads in our 1 layer
d_head  = 32           # size per head
d_mlp   = 512          # MLP hidden width
n_ctx   = 3            # sequence length: [a, b, =]
d_vocab = p + 1        # one token per number 0..p-1 plus a dedicated '=' token

# training
train_frac    = 0.3    # fraction of all (a,b) pairs used for training
lr            = 1e-3
weight_decay  = 1.0    # crucial -- without weight decay, no grokking
betas         = (0.9, 0.98)
n_steps       = 30_000
log_every     = 100    # evaluate train+test loss every N steps

EQUALS_TOKEN = p       # index of the '=' token (we just stuff it at the end of vocab)

## 3. The data

We generate every possible `(a, b)` pair where `a, b ∈ {0, ..., p-1}`. That's `p * p = 12,769` pairs. Each example is a length-3 sequence `[a, b, =]` and a target `(a + b) mod p`. We shuffle and take 30% for training.

Because the dataset is tiny we'll train on the entire train set every step (full-batch). Full-batch training is part of what makes grokking happen cleanly - minibatch noise can prevent the slow circuit-formation we want to see.

In [ ]:
# build all (a, b) pairs and their labels
a = torch.arange(p).repeat_interleave(p)   # 0,0,..,0, 1,1,..,1, ...
b = torch.arange(p).repeat(p)              # 0,1,..,p-1, 0,1,..,p-1, ...
labels = (a + b) % p                       # (p*p,)

# format inputs as sequences [a, b, '=']
inputs = torch.stack([a, b, torch.full_like(a, EQUALS_TOKEN)], dim=1)  # (p*p, 3)
print('inputs shape:', inputs.shape, '  labels shape:', labels.shape)
print('first 3 examples:')
for i in range(3):
    print(f'  {inputs[i].tolist()}  →  {labels[i].item()}')

# shuffle and split
n_total = inputs.shape[0]
perm = torch.randperm(n_total)
inputs = inputs[perm]
labels = labels[perm]
n_train = int(train_frac * n_total)

train_x = inputs[:n_train].to(device)
train_y = labels[:n_train].to(device)
test_x  = inputs[n_train:].to(device)
test_y  = labels[n_train:].to(device)

print(f'\ntrain: {len(train_x):,} examples  |  test: {len(test_x):,} examples')

## 4. The transformer

A 1-layer transformer built by hand. We use raw weight matrices (`WQ, WK, WV, WO, Win, Wout`, etc.) rather than the higher-level `nn.Linear` everywhere, so it's easy to inspect them after training.

The architecture:

```
tokens → embed + positional → attention → MLP → unembed → logits
             (WE, Wpos)                              (W_U)
```

with residual connections around attention and MLP (the standard transformer trick).

### 4a. Attention

Attention moves information between the 3 positions. Each of the 4 heads independently computes queries, keys, and values, and uses them to form a weighted sum across positions. Read this code alongside the attention explainer in the walkthrough above if any of it feels mysterious.

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, n_heads, d_head):
        super().__init__()
        # shapes chosen so we can keep heads as an explicit dim throughout
        self.W_Q = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_K = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_V = nn.Parameter(torch.randn(n_heads, d_model, d_head) * (1.0 / d_model**0.5))
        self.W_O = nn.Parameter(torch.randn(n_heads, d_head, d_model) * (1.0 / d_model**0.5))
        self.d_head = d_head

    def forward(self, x):
        # x: (batch, pos, d_model)
        # einsum letters: b=batch  p,q=positions  h=heads  d=d_model  e=d_head
        q = torch.einsum('bpd,hde->bphe', x, self.W_Q)
        k = torch.einsum('bpd,hde->bphe', x, self.W_K)
        v = torch.einsum('bpd,hde->bphe', x, self.W_V)

        # attention scores: dot product of queries and keys, scaled by sqrt(d_head)
        scores = torch.einsum('bphe,bqhe->bhpq', q, k) / (self.d_head ** 0.5)
        attn   = F.softmax(scores, dim=-1)  # (batch, head, query_pos, key_pos)

        # weighted sum of values, then project back to d_model with W_O
        z   = torch.einsum('bhpq,bqhe->bphe', attn, v)
        out = torch.einsum('bphe,hed->bpd',   z, self.W_O)
        return out

### 4b. MLP

The MLP runs independently at each position. It's just `linear → ReLU → linear`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_model, d_mlp):
        super().__init__()
        self.W_in  = nn.Parameter(torch.randn(d_model, d_mlp) * (1.0 / d_model**0.5))
        self.W_out = nn.Parameter(torch.randn(d_mlp, d_model) * (1.0 / d_mlp**0.5))

    def forward(self, x):
        return F.relu(x @ self.W_in) @ self.W_out

### 4c. Putting it together

Embedding → attention (with residual) → MLP (with residual) → unembedding.

Note we deliberately omit `LayerNorm` - the paper omits it too. It makes the mech interp analysis much cleaner because the residual stream isn't being repeatedly rescaled.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_vocab, d_model, n_heads, d_head, d_mlp, n_ctx):
        super().__init__()
        self.W_E   = nn.Parameter(torch.randn(d_vocab, d_model) * (1.0 / d_model**0.5))
        self.W_pos = nn.Parameter(torch.randn(n_ctx, d_model)   * (1.0 / d_model**0.5))
        self.attn  = Attention(d_model, n_heads, d_head)
        self.mlp   = MLP(d_model, d_mlp)
        self.W_U   = nn.Parameter(torch.randn(d_model, d_vocab) * (1.0 / d_model**0.5))

    def forward(self, tokens):
        # tokens: (batch, n_ctx) of ints in [0, d_vocab)
        residual = self.W_E[tokens] + self.W_pos          # (batch, n_ctx, d_model)
        residual = residual + self.attn(residual)         # residual connection
        residual = residual + self.mlp(residual)          # residual connection
        logits   = residual @ self.W_U                    # (batch, n_ctx, d_vocab)
        return logits

In [ ]:
# instantiate + sanity check
model = Transformer(d_vocab, d_model, n_heads, d_head, d_mlp, n_ctx).to(device)

n_params = sum(p_.numel() for p_ in model.parameters())
print(f'Total parameters: {n_params:,}')

# quick forward to confirm shapes
with torch.no_grad():
    sample_logits = model(train_x[:4])
print('sample logits shape:', tuple(sample_logits.shape), '(should be (4, 3, 114))')

## 5. Training

Full-batch AdamW with weight decay = 1.0. We compute cross-entropy on the logits at the `=` position (the last token) against the true label.

We log train and test loss every 100 steps. Expect this cell to takeon a Colab T4.

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=lr, weight_decay=weight_decay, betas=betas
)

train_losses, test_losses, log_steps = [], [], []

for step in range(n_steps + 1):
    # ---- forward + backward on training set ----
    logits = model(train_x)                              # (n_train, 3, d_vocab)
    train_loss = F.cross_entropy(logits[:, -1, :], train_y)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    # ---- periodic evaluation on the held-out test set ----
    if step % log_every == 0:
        with torch.no_grad():
            test_logits = model(test_x)
            test_loss = F.cross_entropy(test_logits[:, -1, :], test_y)
        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())
        log_steps.append(step)
        if step % 2000 == 0:
            print(f'step {step:6d}  |  train {train_loss.item():.5f}  test {test_loss.item():.5f}')

print('\nTraining complete.')

## 6. The grokking curve - the wow moment

Plot train and test loss on a log scale. The signature should be:

1. Train loss crashes to near zero in the first few hundred steps.
2. Test loss stays near `log(p) ≈ 4.73` (random-guess level) for a long flat plateau.
3. Eventually - somewhere between step 8k and 25k - test loss falls off a cliff.

That cliff is grokking.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(log_steps, train_losses, label='train loss', linewidth=2)
plt.plot(log_steps, test_losses,  label='test loss',  linewidth=2)
plt.axhline(np.log(p), color='gray', linestyle='--', linewidth=1,
            label=f'random-guess level  log(p) = {np.log(p):.2f}')
plt.yscale('log')
plt.xlabel('step')
plt.ylabel('loss (log scale)')
plt.title('Grokking: late, sudden generalization after long memorization plateau')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Stretch: Fourier analysis of the embedding

If the model really learnt a trigonometric algorithm, then the embedding matrix `W_E` should not be random - its rows (one per number `0..p-1`) should look like samples of a small set of sine and cosine waves.

We can check this directly by taking a discrete Fourier transform of `W_E` along the number axis. If the model is using only a few frequencies, the FFT will be sparse - concentrated at those frequencies and near-zero elsewhere.

(The `=` token is not part of this pattern; we drop it for the analysis.)

In [ ]:
# grab embedding for the number tokens only
W_E_nums = model.W_E[:p].detach().cpu().float()   # (p, d_model)

# discrete Fourier transform along the number (vocab) axis
fft = torch.fft.fft(W_E_nums, dim=0)              # (p, d_model), complex
fft_norms = fft.abs()                             # magnitude per (frequency, hidden_dim)

# total 'energy' at each frequency: ||fft_freq||_2 across the d_model axis
energy = fft_norms.norm(dim=1).numpy()            # (p,)

# FFT is symmetric (W_E is real) so freq k and freq p-k have equal energy.
# Only the first p//2 + 1 frequencies are independent.
freqs = np.arange(p // 2 + 1)
energy_first_half = energy[:p // 2 + 1]

plt.figure(figsize=(10, 4))
plt.bar(freqs, energy_first_half, color='C0')
plt.xlabel('frequency k')
plt.ylabel('energy at frequency k (L2 norm across d_model)')
plt.title('Fourier spectrum of the embedding matrix')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# print the top frequencies
top_k = np.argsort(energy_first_half)[::-1][:8]
print('Top frequencies by energy:')
for rank, k in enumerate(top_k):
    print(f'  rank {rank + 1}: k = {k:3d}   energy = {energy_first_half[k]:.3f}')

### What you should see

Most of the bars near zero, with a small number of "spikes" at specific frequencies. Those spikes are the model's chosen frequencies - the model represents each number `x` as `[..., cos(2π·k·x/p), sin(2π·k·x/p), ...]` for `k` in that small set.

The full mech interp story (which we don't replicate in this notebook) is that:

1. The embedding rotates each number `x` into a Fourier basis.
2. The MLP uses its quadratic-ish structure (linear → ReLU → linear) to implement the product-to-sum trig identity `cos(2πk(a+b)/p) = cos(2πka/p)·cos(2πkb/p) − sin(2πka/p)·sin(2πkb/p)`.
3. The attention head shuffles information from positions 0 and 1 (where `a` and `b` live) onto position 2 (the `=`).
4. The unembedding reads out the result, peaked at the correct answer.

What you just discovered (a sparse Fourier spectrum in `W_E`) is the first piece of evidence for step 1. The full circuit reverse-engineering is a great next project - Neel's [grokking demo Colab](https://colab.research.google.com/github/neelnanda-io/Easy-Transformer/blob/main/GrokkingDemo.ipynb) walks through it.

## 8. Discussion

Two big takeaways:

Generalization is not always smooth. The standard ML picture - train loss and test loss decline together - is a property of the typical regime, not a law. Under the right conditions (algorithmic task, small model, weight decay) you can get a long memorisation phase followed by a sharp transition to generalisation. Loss curves can hide structural changes happening underneath.

Models can learn surprisingly elegant algorithms. A 1-layer transformer with no LayerNorm and 200k parameters, trained by SGD on a pile of `(a, b) → (a+b) mod p` examples, rediscovered Fourier series on its own and used the trigonometric product-to-sum identity to do modular arithmetic. Nobody told it about sines and cosines. That's the kind of result that makes mech interp worth doing - the algorithm the network learns is often clean enough that a human can understand it after the fact.

What we didn't do (good follow-ups):

- Compute the paper's three progress measures during training (restricted loss, excluded loss, Gini coefficient). They reveal that the circuit is forming gradually during the plateau, even though the loss curve looks flat.
- Reverse-engineer the attention head and MLP (not just the embedding).
- Try other modular operations (multiplication, group operations) and see what circuits emerge.

See "Where to go next" below for pointers.